%pip install langgraph google-genai langchain-google-genai beautifulsoup4 pyodbc python-dotenv

In [150]:
import os
import re
import email
import imaplib
import pyodbc

from datetime import datetime, timedelta
from email.header import decode_header
from email.utils import parsedate_to_datetime

from bs4 import BeautifulSoup
from google import genai

from dotenv import load_dotenv

from typing import TypedDict

from langgraph.graph import StateGraph, START, END

In [151]:
import os
from dotenv import load_dotenv

load_dotenv()

EMAIL_USER = os.getenv("EMAIL_USER")
EMAIL_PASSWORD = os.getenv("EMAIL_PASS")

print("Email:", EMAIL_USER)
print("Password loaded:", bool(EMAIL_PASSWORD))

Email: abhishek.mukherjee@mpokket.com
Password loaded: True


In [152]:
SQL_SERVER = os.getenv("SQL_SERVER")
SQL_DATABASE = os.getenv("SQL_DATABASE")
SQL_USERNAME = os.getenv("SQL_USERNAME")
SQL_PASSWORD = os.getenv("SQL_PASSWORD")
ai_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("SQL server:", SQL_SERVER)
print("SQL creds loaded:", all([SQL_DATABASE, SQL_USERNAME, SQL_PASSWORD]))

SQL server: 192.168.210.125
SQL creds loaded: True


In [153]:
def decode_subject(subject):

    if not subject:
        return ""

    decoded_parts = decode_header(subject)

    subject_text = ""

    for part, encoding in decoded_parts:

        if isinstance(part, bytes):

            subject_text += part.decode(
                encoding or "utf-8",
                errors="ignore"
            )

        else:

            subject_text += part

    return subject_text

In [154]:
class WorkflowState(TypedDict, total=False):

    # Gmail information
    email_id: str
    subject: str
    sender: str
    email_date: str
    email_body: str

    # Phone
    phone_number: str
    extraction_method: str

    # Database
    external_user_id: str
    user_id: str
    sql_phone: str
    external_user_id: str
    user_id: str
    ref_relation: str
    relation_name: str





    # Final response
    final_reply: str

    # Error
    error: str

In [155]:
def decode_subject(subject):

    if not subject:
        return ""

    decoded_parts = decode_header(subject)

    subject_text = ""

    for part, encoding in decoded_parts:

        if isinstance(part, bytes):

            subject_text += part.decode(
                encoding or "utf-8",
                errors="ignore"
            )

        else:

            subject_text += part

    return subject_text

In [156]:
def extract_email_body(msg):

    body_parts = []

    if msg.is_multipart():

        for part in msg.walk():

            content_type = part.get_content_type()

            content_disposition = str(
                part.get("Content-Disposition", "")
            ).lower()

            # Ignore attachments
            if "attachment" in content_disposition:
                continue

            if content_type in ["text/plain", "text/html"]:

                try:

                    payload = part.get_payload(
                        decode=True
                    )

                    if not payload:
                        continue

                    charset = (
                        part.get_content_charset()
                        or "utf-8"
                    )

                    text = payload.decode(
                        charset,
                        errors="ignore"
                    )

                    body_parts.append(text)

                except Exception as e:

                    print(
                        f"⚠️ Body extraction error: {e}"
                    )

    else:

        try:

            payload = msg.get_payload(
                decode=True
            )

            if payload:

                charset = (
                    msg.get_content_charset()
                    or "utf-8"
                )

                text = payload.decode(
                    charset,
                    errors="ignore"
                )

                body_parts.append(text)

        except Exception as e:

            print(
                f"⚠️ Body extraction error: {e}"
            )

    return "\n".join(body_parts)

In [157]:
def gmail_node(state: WorkflowState):

    print("=" * 70)
    print("NODE 1: GMAIL")
    print("=" * 70)

    mail = None

    try:

        today = datetime.now()
        yesterday = today - timedelta(days=1)

        print(
            f"Looking for emails from: "
            f"{yesterday.strftime('%d-%b-%Y')}"
        )

        mail = imaplib.IMAP4_SSL(
            "imap.gmail.com"
        )

        mail.login(
            EMAIL_USER,
            EMAIL_PASSWORD
        )

        print("✅ Gmail login successful")

        status, data = mail.select(
            "rca",
            readonly=True
        )

        if status != "OK":

            return {
                "error":
                "Could not select rca label"
            }

        search_date = yesterday.strftime(
            "%d-%b-%Y"
        )

        status, messages = mail.search(
            None,
            "SINCE",
            search_date
        )

        if status != "OK":

            return {
                "error":
                "Gmail search failed"
            }

        email_ids = messages[0].split()

        print(
            f"Emails returned by Gmail: "
            f"{len(email_ids)}"
        )

        for e_id in email_ids:

            status, msg_data = mail.fetch(
                e_id,
                "(RFC822)"
            )

            if status != "OK":
                continue

            for response_part in msg_data:

                if not isinstance(
                    response_part,
                    tuple
                ):
                    continue

                msg = email.message_from_bytes(
                    response_part[1]
                )

                date_str = msg.get("Date")

                try:

                    email_datetime = (
                        parsedate_to_datetime(
                            date_str
                        )
                    )

                    if (
                        email_datetime.date()
                        != yesterday.date()
                    ):
                        continue

                except Exception:

                    continue

                subject = decode_subject(
                    msg.get("Subject")
                )

                sender = msg.get("From")

                body = extract_email_body(
                    msg
                )

                print(
                    f"✅ Email found: {subject}"
                )

                return {

                    "email_id":
                    e_id.decode(
                        errors="ignore"
                    ),

                    "subject":
                    subject,

                    "sender":
                    sender,

                    "email_date":
                    date_str,

                    "email_body":
                    body
                }

        return {
            "error":
            "No RCA email found yesterday"
        }

    except Exception as e:

        return {
            "error":
            f"Gmail Error: {e}"
        }

    finally:

        if mail:

            try:
                mail.logout()
            except:
                pass

In [158]:
def fetch_yesterdays_emails():
    mail, emails = None, []
    try:
        yesterday = (datetime.now() - timedelta(days=1)).date()
        since = yesterday.strftime("%d-%b-%Y")
        before = (yesterday + timedelta(days=1)).strftime("%d-%b-%Y")

        mail = imaplib.IMAP4_SSL("imap.gmail.com")
        mail.login(EMAIL_USER, EMAIL_PASSWORD)

        status, _ = mail.select("rca", readonly=True)
        if status != "OK":
            print("Could not select rca label")
            return []

        status, messages = mail.search(None, "SINCE", since, "BEFORE", before)
        if status != "OK":
            return []

        for e_id in messages[0].split():
            status, msg_data = mail.fetch(e_id, "(RFC822)")
            if status != "OK":
                continue

            for part in msg_data:
                if not isinstance(part, tuple):
                    continue

                msg = email.message_from_bytes(part[1])
                date_str = msg.get("Date")

                try:
                    if parsedate_to_datetime(date_str).astimezone().date() != yesterday:
                        continue
                except Exception:
                    continue

                body = extract_email_body(msg)

                flat = " ".join(body.lower().split())   # fixes line-wrapped text
                if "confirm whether the mentioned number" not in flat:
                    continue                             # skip replies like "we did not find any link"

                emails.append({
                    "email_id": e_id.decode(errors="ignore"),
                    "subject": decode_subject(msg.get("Subject")),
                    "sender": msg.get("From"),
                    "email_date": date_str,
                    "email_body": body,
                    # extra fields for creating a threaded reply
                    "message_id": msg.get("Message-ID"),
                    "reply_to": msg.get("Reply-To"),
                    "to": msg.get("To"),
                    "cc": msg.get("Cc"),
                    "references": msg.get("References"),
                })

        return emails

    except Exception as e:
        print("Gmail Error:", e)
        return emails

    finally:
        if mail:
            try:
                mail.logout()
            except Exception:
                pass

In [159]:
def phone_extraction_node(state: WorkflowState):

    print("\n")
    print("=" * 70)
    print("NODE 2: PHONE EXTRACTION")
    print("=" * 70)

    email_body = state.get(
        "email_body",
        ""
    )

    if not email_body:

        return {
            "error":
            "Email body is empty"
        }

    # --------------------------------------------------
    # BeautifulSoup
    # --------------------------------------------------

    soup = BeautifulSoup(
        email_body,
        "html.parser"
    )

    clean_text = soup.get_text(
        " ",
        strip=True
    )

    # --------------------------------------------------
    # Regex
    # --------------------------------------------------

    phones = re.findall(
        r"(?<!\d)[6-9]\d{9}(?!\d)",
        clean_text
    )

    if phones:

        phone = phones[0]

        print(
            f"✅ Phone found using Regex: "
            f"{phone}"
        )

        return {

            "phone_number":
            phone,

            "extraction_method":
            "BeautifulSoup + Regex"
        }

    # --------------------------------------------------
    # Gemini fallback
    # --------------------------------------------------

    print(
        "⚠️ Regex failed."
    )

    print(
        "➡ Calling Gemini..."
    )

    prompt = f"""
Read this RCA email.

Find the target Indian 10-digit mobile number.

Return ONLY the 10 digit number.

If no number is present, return NONE.

Email:

{clean_text}
"""

    try:

        response = ai_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )

        result = response.text.strip()

        match = re.search(
            r"(?<!\d)[6-9]\d{9}(?!\d)",
            result
        )

        if match:

            phone = match.group(0)

            print(
                f"✅ Phone found by Gemini: "
                f"{phone}"
            )

            return {

                "phone_number":
                phone,

                "extraction_method":
                "Gemini"
            }

        return {
            "error":
            "Gemini could not find phone number"
        }

    except Exception as e:

        return {
            "error":
            f"Gemini Error: {e}"
        }

In [160]:
def sql_node(state: WorkflowState):

    print("\n")
    print("=" * 70)
    print("NODE 3: SQL DATABASE")
    print("=" * 70)

    phone = state.get("phone_number")

    if not phone:
        return {
            "error": "Phone number not available"
        }

    # Add 112 before checking SQL
    sql_phone = "112" + str(phone)

    print(f"Extracted Phone : {phone}")
    print(f"SQL Phone       : {sql_phone}")

    conn = None

    try:

        conn_str = (
            "DRIVER={ODBC Driver 17 for SQL Server};"
            f"SERVER={SQL_SERVER};"
            f"DATABASE={SQL_DATABASE};"
            f"UID={SQL_USERNAME};"
            f"PWD={SQL_PASSWORD};"
        )

        conn = pyodbc.connect(conn_str)
        cursor = conn.cursor()

        # Query 1
        query_1 = """
        SELECT
            [user_id],
            [refRelation]
        FROM [Dialer_Data].[dbo].[Mpokket_Dialer_Dump]
        WHERE [contact_info] = ?
          AND ISNULL([refRelation], 'Others') <> 'P'
        """

        cursor.execute(query_1, (sql_phone,))

        rows = cursor.fetchall()

        if not rows:
            return {
                "sql_phone": sql_phone,
                "error": f"Phone {sql_phone} not found with a valid relation"
            }

        print(f"Found {len(rows)} matching record(s)")

        for row in rows:

            external_user_id = str(row[0])
            ref_relation = str(row[1]) if row[1] else "Others"

            print(
                f"External User ID: {external_user_id} | "
                f"Relation: {ref_relation}"
            )

            # Query 2
            query_2 = f"""
            SELECT [user_id]
                 FROM [{SQL_DATABASE}].[dbo].[overdue_users_report_Analytics]
                 WHERE [external_user_id] = ?
             """

            cursor.execute(query_2, (external_user_id,))

            row2 = cursor.fetchone()

            if row2:

                user_id = str(row2[0])

                print(f"Final User ID: {user_id}")

                if ref_relation == "A":
                    relation_name = "alternate"
                elif ref_relation == "V-Bureau":
                    relation_name = "Bureau"
                elif ref_relation == "others":
                    relation_name = "Reference"    
                else:
                    relation_name = "other"

                return {
                    "sql_phone": sql_phone,
                    "external_user_id": external_user_id,
                    "user_id": user_id,
                    "ref_relation": ref_relation,
                    "relation_name": relation_name
                }

        return {
            "sql_phone": sql_phone,
            "error": (
                f"Phone {sql_phone} found, but no matching "
                "Analytics record was found"
            )
        }

    except Exception as e:

        return {
            "sql_phone": sql_phone,
            "error": f"SQL Error: {e}"
        }

    finally:

        if conn:
            conn.close()

In [165]:
import json

LLM_MODEL = "gemini-2.5-flash"
PHONE_PATTERN = r"(?<!\d)[6-9]\d{9}(?!\d)"


def ask_llm_json(prompt):
    response = ai_client.models.generate_content(
        model=LLM_MODEL,
        contents=prompt,
        config={"response_mime_type": "application/json", "temperature": 0},
    )
    return json.loads(response.text)


def clean_email_text(email_body):
    text = BeautifulSoup(email_body, "html.parser").get_text(" ", strip=True)
    top = re.split(r"-{3,}\s*Forwarded message", text, maxsplit=1)[0]
    return top or text


def select_target_numbers(email_text, candidates):
    """Ask the LLM which of the candidate numbers the requester is asking about."""

    if len(candidates) <= 1:
        return candidates

    prompt = f"""
You are reading an internal RCA request email. The requester asks whether a
customer's mobile number is linked to any other user account.

Candidate 10-digit numbers found in the email: {candidates}

Return JSON exactly like {{"numbers": ["9876543210"]}} containing ONLY the numbers
the requester wants checked. Exclude support or helpline numbers, employee or agent
numbers, and numbers that only appear in email signatures.
Only use numbers from the candidate list.

Email:
{email_text[:6000]}
"""

    try:
        data = ask_llm_json(prompt)
        numbers = data.get("numbers", []) if isinstance(data, dict) else data

        # Guardrail: the model may only pick numbers that were really in the email
        chosen = [n for n in dict.fromkeys(numbers) if n in candidates]

        return chosen if chosen else candidates      # if unsure, keep everything

    except Exception as e:
        print("⚠️ LLM number selection failed, using all numbers:", e)
        return candidates


def llm_is_request(email_text):
    """Classify an email that did not match the phrase filter."""

    prompt = f"""
Classify this internal email.

"request": someone asks whether a mobile number is linked to any other user
           account, or asks to check or identify a number.
"reply":   someone answers such a request.
"other":   anything else.

Return JSON exactly like {{"type": "request"}}.

Email:
{email_text[:4000]}
"""

    try:
        data = ask_llm_json(prompt)
        return isinstance(data, dict) and data.get("type") == "request"

    except Exception as e:
        print("⚠️ LLM classification failed, skipping email:", e)
        return False

In [166]:
def reply_node(state: WorkflowState):

    print("\n")
    print("=" * 70)
    print("NODE 4: DRAFT REPLY")
    print("=" * 70)

    greeting_name = "Rinita"

    user_id = state.get("user_id")
    relation = state.get("relation_name") or "other"
    error = state.get("error")

    # Errors from sql_node that really mean "no link found"
    NOT_FOUND_MARKERS = (
        "not found with a valid relation",
        "no matching Analytics record",
    )

    # ==========================================================
    # ERROR / NOT FOUND
    # ==========================================================

    if error:

        if any(marker in error for marker in NOT_FOUND_MARKERS):

            reply = (
                f"Hi {greeting_name},\n\n"
                "We did not find any link to another number."
            )

        else:

            # Real failure (SQL, Gmail, phone extraction): don't draft a wrong answer
            reply = (
                "⚠️ MANUAL CHECK NEEDED - no reply drafted.\n"
                f"Reason: {error}"
            )

    # ==========================================================
    # FOUND (alternate / beaureau / other)
    # ==========================================================

    else:

        reply = (
            f"Hi {greeting_name},\n\n"
            f"This is to inform you that the mentioned number belongs to the "
            f"{relation} contact of the below-mentioned user.\n\n"
            f"User ID: {user_id}"
        )

    print("\nDraft Reply:")
    print("-" * 70)
    print(reply)

    return {
        "final_reply": reply
    }

In [167]:
def route_start(state: WorkflowState):
    return "sql" if state.get("phone_number") else "extract_phone"


workflow = StateGraph(WorkflowState)

workflow.add_node("extract_phone", phone_extraction_node)
workflow.add_node("sql", sql_node)
workflow.add_node("reply", reply_node)

workflow.add_conditional_edges(
    START,
    route_start,
    {"sql": "sql", "extract_phone": "extract_phone"},
)
workflow.add_edge("extract_phone", "sql")
workflow.add_edge("sql", "reply")
workflow.add_edge("reply", END)

app = workflow.compile()

print("Graph compiled successfully!")

Graph compiled successfully!


In [168]:
from pathlib import Path


def extract_all_phones(email_body):
    text = BeautifulSoup(email_body, "html.parser").get_text(" ", strip=True)

    # Only the top of the email, before the forwarded chain
    top = re.split(r"-{3,}\s*Forwarded message", text, maxsplit=1)[0]

    found = re.findall(PHONE_PATTERN, top) or re.findall(PHONE_PATTERN, text)
    candidates = list(dict.fromkeys(found))     # unique, keep order

    return select_target_numbers(top or text, candidates)


emails = fetch_yesterdays_emails()
print(f"RCA request emails from yesterday: {len(emails)}")

results = []      # list of (email, result)

for em in emails:

    phones = extract_all_phones(em["email_body"])

    print("\n" + "#" * 70)
    print("EMAIL:", em["subject"])
    print("Numbers found:", phones if phones else "none (Gemini fallback will be tried)")
    print("#" * 70)

    for phone in (phones if phones else [None]):

        state = dict(em)

        if phone:
            state["phone_number"] = phone
            state["extraction_method"] = "Regex (multi-number)"

        result = app.invoke(state)
        results.append((em, result))


# ---------- Print all drafts together ----------
print("\n" + "=" * 70)
print("ALL DRAFT REPLIES")
print("=" * 70)

for em, res in results:
    print(f"\n### {em['subject']}")
    print(f"Number: {res.get('phone_number')}")
    print(res.get("final_reply"))
    print("-" * 70)


# ---------- Save to txt file ----------
out_dir = Path("rca_replies")
out_dir.mkdir(exist_ok=True)

reply_date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
out_file = out_dir / f"rca_replies_{reply_date}.txt"

SEP = "-" * 70
ready, manual = [], []

for em, res in results:
    reply = (res.get("final_reply") or "").strip()
    if not reply or reply.startswith("⚠️"):
        manual.append((em, res, reply))
    else:
        ready.append((em, res, reply))

with open(out_file, "w", encoding="utf-8") as f:

    f.write(f"RCA REPLIES FOR {reply_date}\n")
    f.write(f"Ready to send: {len(ready)} | Need manual check: {len(manual)}\n")
    f.write("=" * 70 + "\n\n")

    for em, res, reply in ready:
        f.write(f"### {em['subject']}\n")
        f.write(f"Number: {res.get('phone_number')}\n\n")
        f.write(reply + "\n\n")
        f.write(SEP + "\n\n")

    if manual:
        f.write("NEED MANUAL CHECK (do not send)\n")
        f.write("=" * 70 + "\n\n")
        for em, res, reply in manual:
            f.write(f"### {em['subject']}\n")
            f.write(f"Number: {res.get('phone_number')}\n")
            f.write(f"{reply or 'No reply drafted'}\n\n")
            f.write(SEP + "\n\n")

print("\nSaved to:", out_file.resolve())

RCA request emails from yesterday: 8

######################################################################
EMAIL: Fwd: Calls Received for Another Person's Loan // 20685177
Numbers found: ['7981080394']
######################################################################


NODE 3: SQL DATABASE
Extracted Phone : 7981080394
SQL Phone       : 1127981080394


NODE 4: DRAFT REPLY

Draft Reply:
----------------------------------------------------------------------
Hi Rinita,

We did not find any link to another number.

######################################################################
EMAIL: Fwd: RCA||bijoycb8590@gmail.com
Numbers found: ['9846581873']
######################################################################


NODE 3: SQL DATABASE
Extracted Phone : 9846581873
SQL Phone       : 1129846581873


NODE 4: DRAFT REPLY

Draft Reply:
----------------------------------------------------------------------
Hi Rinita,

We did not find any link to another number.

##################

In [171]:
import json
import time
from pathlib import Path
from email.message import EmailMessage
from email.utils import getaddresses, format_datetime

DRAFTS_FOLDER = '"[Gmail]/Drafts"'      # Gmail's default English name
LOG_FILE = Path("rca_replies") / "drafted_log.json"
LOG_FILE.parent.mkdir(exist_ok=True)


def clean(value):
    return " ".join((value or "").split())      # removes line breaks inside headers


def load_log():
    if LOG_FILE.exists():
        try:
            return set(json.loads(LOG_FILE.read_text(encoding="utf-8")))
        except Exception:
            return set()
    return set()


def save_log(keys):
    LOG_FILE.write_text(json.dumps(sorted(keys), indent=2), encoding="utf-8")


def build_reply_message(em, reply_text):
    me = EMAIL_USER.lower()

    # Reply-all: To = Reply-To (or From), Cc = original To/Cc, minus myself
    to_addrs = []
    for _, addr in getaddresses([clean(em.get("reply_to") or em.get("sender"))]):
        if addr and addr.lower() != me and addr.lower() not in [a.lower() for a in to_addrs]:
            to_addrs.append(addr)

    cc_addrs = []
    for _, addr in getaddresses([clean(em.get("to")), clean(em.get("cc"))]):
        taken = [a.lower() for a in to_addrs + cc_addrs]
        if addr and addr.lower() != me and addr.lower() not in taken:
            cc_addrs.append(addr)

    if not to_addrs:
        raise ValueError("No recipient found in original email")

    subject = clean(em["subject"])
    if not subject.lower().startswith("re:"):
        subject = "Re: " + subject

    msg = EmailMessage()
    msg["From"] = EMAIL_USER
    msg["To"] = ", ".join(to_addrs)
    if cc_addrs:
        msg["Cc"] = ", ".join(cc_addrs)
    msg["Subject"] = subject
    msg["Date"] = format_datetime(datetime.now().astimezone())

    original_id = clean(em.get("message_id"))
    if original_id:
        msg["In-Reply-To"] = original_id
        msg["References"] = (clean(em.get("references")) + " " + original_id).strip()

    msg.set_content(reply_text)
    return msg


done = load_log()
created = skipped = failed = 0

mail = imaplib.IMAP4_SSL("imap.gmail.com")
mail.login(EMAIL_USER, EMAIL_PASSWORD)

try:
    for em, res in results:

        reply = (res.get("final_reply") or "").strip()

        # never draft manual-check items
        if not reply or reply.startswith("⚠️"):
            continue

        key = f"{em.get('message_id') or em['email_id']}|{res.get('phone_number')}"

        if key in done:
            skipped += 1
            continue

        try:
            msg = build_reply_message(em, reply)

            status, _ = mail.append(
                DRAFTS_FOLDER,
                "\\Draft",
                imaplib.Time2Internaldate(time.time()),
                msg.as_bytes(),
            )

            if status == "OK":
                done.add(key)
                created += 1
                print(f"✅ Draft created: {res.get('phone_number')} | {em['subject']}")
            else:
                failed += 1
                print(f"❌ Draft failed: {res.get('phone_number')} | {em['subject']}")

        except Exception as e:
            failed += 1
            print(f"❌ Draft error for {res.get('phone_number')}: {e}")

finally:
    try:
        mail.logout()
    except Exception:
        pass
    save_log(done)

print(f"\nDrafts created: {created} | Already drafted (skipped): {skipped} | Failed: {failed}")

✅ Draft created: 7981080394 | Fwd: Calls Received for Another Person's Loan // 20685177
✅ Draft created: 9846581873 | Fwd: RCA||bijoycb8590@gmail.com
✅ Draft created: 6369852153 | Fwd: RCA || Calling frequency || Ticket- 03DJYW3AQ34UEGFX
✅ Draft created: 7678613066 | Fwd: RCA ll 7678613066
✅ Draft created: 9489141974 | Fwd: RCA || 11522486
✅ Draft created: 9445021530 | Re: RCA || 11522486
✅ Draft created: 9489141974 | Re: RCA || 11522486
✅ Draft created: 9318352433 | Fwd: Calling frequency || 9318352433 || 03DJYW3AQ7K7TSUN
✅ Draft created: 9572636353 | Fwd: Non user getting calls for user // 9572636353

Drafts created: 9 | Already drafted (skipped): 0 | Failed: 0


In [169]:
# result = app.invoke({})

# print("\n")
# print("=" * 70)
# print("FINAL STATE")
# print("=" * 70)

# print("Phone:", result.get("phone_number"))
# print("Extraction:", result.get("extraction_method"))
# print("SQL Phone:", result.get("sql_phone"))
# print("Relation:", result.get("ref_relation"))
# print("Relation Name:", result.get("relation_name"))
# print("External User ID:", result.get("external_user_id"))
# print("User ID:", result.get("user_id"))

# print("\nFinal Reply:")
# print(result.get("final_reply"))

# if result.get("error"):
#     print("\nError:")
#     print(result.get("error"))

In [170]:
gmail_test = gmail_node({})

print("=" * 70)
print("GMAIL RESULT")
print("=" * 70)

print("Error:", gmail_test.get("error"))
print("Subject:", gmail_test.get("subject"))
print("Sender:", gmail_test.get("sender"))
print("Body length:", len(gmail_test.get("email_body", "")))

print("\nBODY:")
print(gmail_test.get("email_body", "")[:3000])

NODE 1: GMAIL
Looking for emails from: 28-Sep-2026
✅ Gmail login successful
Emails returned by Gmail: 12
✅ Email found: Fwd: Calls Received for Another Person's Loan // 20685177
GMAIL RESULT
Error: None
Subject: Fwd: Calls Received for Another Person's Loan // 20685177
Sender: Rinita Rudra <rinita.rudra@mpokket.com>
Body length: 15884

BODY:
Hi Gour,

Kindly confirm whether the mentioned number is linked to any other account
or not.

No. 7981080394

---------- Forwarded message ---------
From: Debanjita Debnath <debanjita.debnath@mpokket.com>
Date: Sun, Sep 27, 2026 at 8:00 PM
Subject: Fwd: Calls Received for Another Person's Loan // 20685177
To: RCA Ops <rca-ops@mpokket.com>
Cc: ert coordination <ert.coordination@mpokket.com>


Hi Team,

Please assist.




*Thanks and RegardsDebanjita DebnathTeam Lead - Core Ops -
Grievance_CEmPokket || Customer Support*

www.mpokket.in
[image: https://www.facebook.com/mpokket]
<https://www.facebook.com/mpokket> [image:
https://www.instagram.com/mpokk